In [1]:
tabla='qmcqx10'
dim='dim_cqxcenqx'

In [2]:
import re

cadena = """
    oricenasicod character(1) COLLATE pg_catalog."default",
    cenasicod character(3) COLLATE pg_catalog."default",
    cenquicod character(2) COLLATE pg_catalog."default",
    cenquides character(30) COLLATE pg_catalog."default",
    cenquitipalmcod character(1) COLLATE pg_catalog."default",
    cenquialmcod character(2) COLLATE pg_catalog."default"
"""

# Reemplaza los caracteres innecesarios
cadena = cadena.replace(" COLLATE pg_catalog.\"default\",", "")
cadena = cadena.replace(" with time zone", "")

# Divide la cadena en una lista de líneas
lineas = cadena.split("\n")

# Crea la cadena de alteración de columnas
cadena_alter = ""
for i, linea in enumerate(lineas):
    palabras = linea.split()
    if len(palabras) >= 2:
        columna = palabras[0]
        tipo = palabras[1]
        if i == len(lineas) - 2:
            # Última línea, agrega punto y coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo};\n"
        else:
            # Otras líneas, agrega coma
            cadena_alter += f"ALTER COLUMN {columna} TYPE {tipo},\n"

# Utiliza una expresión regular para eliminar las comas duplicadas
cadena_alter = re.sub(r',+$', ',', cadena_alter, flags=re.MULTILINE)

print(cadena_alter)



import re

nombrecitos = re.findall(r'ALTER COLUMN\s+(\S+)', cadena_alter)
insertado_col = ','.join(nombrecitos)

print(insertado_col)

ALTER COLUMN oricenasicod TYPE character(1),
ALTER COLUMN cenasicod TYPE character(3),
ALTER COLUMN cenquicod TYPE character(2),
ALTER COLUMN cenquides TYPE character(30),
ALTER COLUMN cenquitipalmcod TYPE character(1),
ALTER COLUMN cenquialmcod TYPE character(2);

oricenasicod,cenasicod,cenquicod,cenquides,cenquitipalmcod,cenquialmcod


In [3]:
tabla='qmcqx10'

import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
import decouple

config = decouple.AutoConfig(' ')

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [4]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527
engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)
connection2.close()

In [5]:
base2.head(176)

,oricenasicod,cenasicod,cenquicod,cenquides,cenquitipalmcod,cenquialmcod
0,1,265,01,CENTRO QUIRURGICO GENERAL,1,24
1,1,308,01,CENTRO QUIRURGICO GENERAL,,
2,1,015,02,CIRUGIA DE DIA,,
3,1,153,01,CENTRO QUIRURGICO,1,01
4,1,015,01,CENTRO QUIRURGICO,1,04
...,...,...,...,...,...,...
170,1,392,07,RENAINCOR,1,08
171,1,005,UV,SALA IMAGENES,1,23
172,1,111,07,SALA # 7 CIRUGIA MENOR,6,04
173,1,079,03,CENTRO QUIRURGICO E.ESCOMEL,1,03


In [6]:
base2.columns

Index(['oricenasicod', 'cenasicod', 'cenquicod', 'cenquides',
       'cenquitipalmcod', 'cenquialmcod'],
      dtype='object')

In [7]:
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

query_count_before = f"SELECT COUNT(*) FROM {tabla}"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} antes de la inserción: {cant_antes}")

base2.to_sql(name=f'tmp_{tabla}', con=engine3, if_exists='replace', index=False)

Cantidad de filas en la tabla qmcqx10 antes de la inserción: 175


175

In [8]:

baseRespaldo=base2.copy()
baseRespaldo.head()

,oricenasicod,cenasicod,cenquicod,cenquides,cenquitipalmcod,cenquialmcod
0,1,265,01,CENTRO QUIRURGICO GENERAL,1,24
1,1,308,01,CENTRO QUIRURGICO GENERAL,,
2,1,015,02,CIRUGIA DE DIA,,
3,1,153,01,CENTRO QUIRURGICO,1,01
4,1,015,01,CENTRO QUIRURGICO,1,04


In [9]:
query=f"""
ALTER TABLE tmp_{tabla}

{cadena_alter}
UPDATE {tabla} b

SET

cenquides = t.cenquides,
cenquitipalmcod = t.cenquitipalmcod,
cenquialmcod = t.cenquialmcod

FROM tmp_{tabla} t
WHERE

b.oricenasicod = t.oricenasicod and b.oricenasicod IS NOT NULL AND
b.cenasicod = t.cenasicod and b.cenasicod IS NOT NULL AND
b.cenquicod = t.cenquicod and b.cenquicod IS NOT NULL;

INSERT INTO {tabla}

({insertado_col})
SELECT 
{insertado_col}

FROM tmp_{tabla}  t
WHERE NOT EXISTS (
    SELECT 1
    FROM {tabla} b 
    WHERE 
    
b.oricenasicod = t.oricenasicod and b.oricenasicod IS NOT NULL AND
b.cenasicod = t.cenasicod and b.cenasicod IS NOT NULL AND
b.cenquicod = t.cenquicod and b.cenquicod IS NOT NULL
) ;
"""

c1= text(query)
connection3.execute(c1)

In [10]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
"""

c2= text(query2)
cursor=connection3.execute(c2)


In [11]:
query_count_after = f"SELECT COUNT(*) FROM {tabla}"
cant_despues = connection3.execute(query_count_after).fetchone()[0]
print(f"Cantidad de filas en la tabla {tabla} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla qmcqx10 después de la inserción: 175
La cantidad de filas insertadas fue de: 0


In [12]:
base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection3)

In [13]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

175

In [14]:
#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

Cantidad de filas en la tabla dim_cqxcenqx antes de la inserción: 175


In [15]:
query=f"""

ALTER TABLE tmp_{tabla} 
{cadena_alter}
INSERT INTO {dim} 

(ori_cas,cod_cas,cod_cqx,des_cqx,tip_alm,cod_alm) 

SELECT

oricenasicod,cenasicod,cenquicod,cenquides,cenquitipalmcod,cenquialmcod

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d
    WHERE 
    
    d.ori_cas = t.oricenasicod AND
    d.cod_cas = t.cenasicod AND
    d.cod_cqx = t.cenquicod

);
"""

c1= text(query)
cursor=connection4.execute(c1)


In [16]:
#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

In [17]:
c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_cqxcenqx después de la inserción: 175
La cantidad de filas insertadas fue de: 0


In [18]:
connection3.close()
connection4.close()